# Aesthetic AI — Kaggle Training Notebook

**Prerequisites:**
- Add your `aesthetic-pairs-queue` Kaggle Dataset as input (contains `queue.db`)
- Enable GPU accelerator: T4 x1
- Run cells **one at a time** and verify each passes before proceeding

In [ ]:
# Cell 1: Clone repo + install deps

GITHUB_REPO = "https://github.com/krutckwang/aesthetic-ai.git"
REPO_DIR = "/kaggle/working/aesthetic-ai"

import os, subprocess

# Step 1: Fix broken Pillow BEFORE importing torchvision.
# Kaggle's base image can have a mixed Pillow state (ImageFont.py from 9.x,
# _util.py from 10.x+). torchvision imports PIL.ImageFont at load time, so
# importing torchvision will crash unless Pillow is first restored to a
# consistent single-version install.
subprocess.run(['pip', 'install', '-q', '--force-reinstall', 'Pillow'], check=True)

# Step 2: Now safe to import torchvision. Capture exact Kaggle versions
# (e.g. 2.10.0+cu128) before pip runs and potentially overwrites them.
import torch, torchvision
from PIL import Image as _PIL

# Step 3: Write constraints file — pins torch, torchvision, and the freshly
# reinstalled Pillow so the main pip install below cannot touch any of them.
with open('/tmp/kaggle_constraints.txt', 'w') as f:
    f.write(f"torch=={torch.__version__}\n")
    f.write(f"torchvision=={torchvision.__version__}\n")
    f.write(f"Pillow=={_PIL.__version__}\n")

# Step 4: Clone / update repo
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

# Step 5: Install project deps — constraints prevent torch/torchvision/Pillow downgrade
subprocess.run([
    'pip', 'install', '-q',
    '-c', '/tmp/kaggle_constraints.txt',
    '--upgrade-strategy', 'only-if-needed',
    'diffusers>=0.27.0', 'peft>=0.11.0', 'accelerate>=0.30.0',
    'insightface>=0.7.3', 'onnxruntime>=1.18.0', 'mediapipe>=0.10.14',
    'sqlalchemy>=2.0.30', 'alembic>=1.13.1', 'facenet-pytorch>=2.5.3',
    'loguru', 'tqdm', 'pyyaml', 'python-dotenv',
], check=True)

print(f"torch:       {torch.__version__}   (must contain +cu128)")
print(f"torchvision: {torchvision.__version__}")
print(f"Pillow:      {_PIL.__version__}")

In [ ]:
# Cell 2: Verify environment
# Expected: CUDA=True, VRAM >= 15 GB. Raise early if GPU not attached.

import torch, torchvision
from PIL import Image

print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Pillow:      {Image.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:         {props.name}")
    print(f"VRAM:        {props.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected — enable T4 accelerator in Notebook settings")

In [ ]:
# Cell 3: Download images from queue.db
# Reads the staging queue exported from the crawler, downloads before/after pairs.
# Skips already-downloaded files. Parses metadata for treatment labels.
# NOTE: Kaggle dataset path format is /kaggle/input/{dataset-slug}/filename

import sqlite3, httpx, json, os, sys, shutil, time, random
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '/kaggle/working/aesthetic-ai')

QUEUE_DB_SRC = "/kaggle/input/aesthetic-pairs-queue/queue.db"
shutil.copy2(QUEUE_DB_SRC, '/kaggle/working/queue.db')

conn = sqlite3.connect('/kaggle/working/queue.db')
rows = conn.execute(
    "SELECT id, before_url, after_url, source_name, metadata "
    "FROM staging_queue WHERE status='pending'"
).fetchall()
conn.close()

IMAGE_DIR = Path('/kaggle/working/aesthetic-ai/data/raw')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

existing = sum(
    1 for r in rows
    if (IMAGE_DIR / f"{r[0]}_before.jpg").exists()
    and (IMAGE_DIR / f"{r[0]}_after.jpg").exists()
)
print(f"Found {len(rows)} pairs in queue — {existing} already downloaded, skipping those")

downloaded, skipped = 0, 0
with httpx.Client(
    timeout=30, follow_redirects=True, verify=False,
    headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
) as client:
    for row_id, before_url, after_url, source_name, metadata_str in tqdm(rows):
        b_path = IMAGE_DIR / f"{row_id}_before.jpg"
        a_path = IMAGE_DIR / f"{row_id}_after.jpg"
        try:
            if not b_path.exists():
                resp = client.get(before_url); resp.raise_for_status()
                b_path.write_bytes(resp.content)
            if not a_path.exists():
                resp = client.get(after_url); resp.raise_for_status()
                a_path.write_bytes(resp.content)
            downloaded += 1
            time.sleep(random.uniform(0.3, 0.8))
        except Exception:
            skipped += 1

print(f"Downloaded: {downloaded}  Failed/skipped: {skipped}")

In [ ]:
# Cell 4: Build manifest.json
# Only includes pairs where both files exist (failed downloads excluded).
# Extracts treatment_category and treatment_brand from metadata JSON so that
# build_instruction() produces specific prompts instead of the generic fallback.

import sqlite3, json
from pathlib import Path

IMAGE_DIR = Path('/kaggle/working/aesthetic-ai/data/raw')
MANIFEST_PATH = Path('/kaggle/working/aesthetic-ai/data/manifest.json')
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect('/kaggle/working/queue.db')
rows = conn.execute(
    "SELECT id, before_url, after_url, source_name, metadata "
    "FROM staging_queue WHERE status='pending'"
).fetchall()
conn.close()

records = []
for row_id, before_url, after_url, source_name, metadata_str in rows:
    b_path = IMAGE_DIR / f"{row_id}_before.jpg"
    a_path = IMAGE_DIR / f"{row_id}_after.jpg"
    if not (b_path.exists() and a_path.exists()):
        continue

    meta = {}
    try:
        meta = json.loads(metadata_str) if metadata_str else {}
    except Exception:
        pass

    records.append({
        "pair_id":            row_id,
        "before_path":        str(b_path),
        "after_path":         str(a_path),
        "treatment_category": meta.get("treatment_category") or meta.get("category"),
        "treatment_brand":    meta.get("treatment_brand")    or meta.get("brand"),
        "zone_codes":         meta.get("zone_codes", []),
    })

MANIFEST_PATH.write_text(json.dumps(records, indent=2), encoding="utf-8")
print(f"Manifest written: {len(records)} pairs → {MANIFEST_PATH}")

labeled = sum(1 for r in records if r["treatment_category"])
print(f"Treatment label coverage: {labeled}/{len(records)} ({100*labeled/max(len(records),1):.1f}%)")
if len(records) == 0:
    raise RuntimeError("Manifest is empty — check Cell 3 output for download errors")

In [ ]:
# Cell 5: Train InstructPix2Pix + LoRA
# batch_size=2 (not 4): IP2P at 512x512 fp16 peaks ~14GB on batch 4, OOMs on T4.
# num_workers=0: multiprocessing DataLoader deadlocks inside Kaggle notebooks.
# Effective batch = 2 (gradient_accumulation_steps defaults to 1 in TrainerConfig).

import os
os.chdir('/kaggle/working/aesthetic-ai')

!python model/training/train.py \
    --manifest        data/manifest.json \
    --base_model      timbrooks/instruct-pix2pix \
    --output_dir      /kaggle/working/lora_output \
    --num_steps       15000 \
    --batch_size      2 \
    --mixed_precision fp16 \
    --lora_rank       16 \
    --save_every      500 \
    --num_workers     0